#                Lab 4  Basic LangchainLLM and Quickstart agent

### For this LangChain agent I followed the tutorial but i did a different example for the tests

## Installations

In [15]:
%pip install python-dotenv
%pip install google-generativeai
%pip install langchain
%pip install langchain-google-genai
%pip install tiktoken

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## Previous API-key configuration

In [16]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    print("ERROR: Crea archivo .env con GOOGLE_API_KEY=tu-key-de-google")
else:
    print("API Key de Google cargada correctamente")

API Key de Google cargada correctamente


##  Initial available Models for LangChain
 

In [17]:
import google.generativeai as genai

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain.tools import tool

genai.configure(api_key=api_key)

print("Modelos disponibles:")
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(f"  - {m.name}")

c:\Users\Manuel Alejandro\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Manuel Alejandro\AppData\Local\Temp\ipykernel_28468\4081685974.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Modelos disponibles:
  - models/gemini-2.5-flash
  - models/gemini-2.5-pro
  - models/gemini-2.0-flash
  - models/gemini-2.0-flash-001
  - models/gemini-2.0-flash-exp-image-generation
  - models/gemini-2.0-flash-lite-001
  - models/gemini-2.0-flash-lite
  - models/gemini-2.5-flash-preview-tts
  - models/gemini-2.5-pro-preview-tts
  - models/gemma-3-1b-it
  - models/gemma-3-4b-it
  - models/gemma-3-12b-it
  - models/gemma-3-27b-it
  - models/gemma-3n-e4b-it
  - models/gemma-3n-e2b-it
  - models/gemini-flash-latest
  - models/gemini-flash-lite-latest
  - models/gemini-pro-latest
  - models/gemini-2.5-flash-lite
  - models/gemini-2.5-flash-image
  - models/gemini-2.5-flash-lite-preview-09-2025
  - models/gemini-3-pro-preview
  - models/gemini-3-flash-preview
  - models/gemini-3.1-pro-preview
  - models/gemini-3.1-pro-preview-customtools
  - models/gemini-3-pro-image-preview
  - models/nano-banana-pro-preview
  - models/gemini-robotics-er-1.5-preview
  - models/gemini-2.5-computer-use-prev

## Basics imports

In [18]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain.tools import tool

## Creation of weather tool

In [19]:
@tool
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

print(get_weather.invoke({"city": "sf"}))

It's always sunny in sf!


## Basic Model Configuration


In [20]:
model = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0.5,
    timeout=10,
    max_tokens=1000,
    google_api_key=api_key
)

print("Modelo configurado: gemini-2.0-flash")

Modelo configurado: gemini-2.0-flash


## Create Basic Agent

In [21]:
agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

print("Agente básico creado")

Agente básico creado


## Creating Real World Agent

### Define system pront with a different topic

In [22]:
SYSTEM_PROMPT = """You are an expert restaurant recommender, who speaks in food puns.

You have access to two tools:

- get_restaurant_recommendation: use this to get restaurant suggestions for a specific city and cuisine
- get_user_cuisine_preference: use this to get the user's preferred cuisine type

If a user asks you for restaurant recommendations, make sure you know their cuisine preference. If you can tell from the question that they mean whatever they like, use the get_user_cuisine_preference tool to find their preference first.

Always respond with food-related puns and wordplay."""

### Other imports

In [23]:
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime
from typing import Optional

### Create tools for the new topic

In [24]:
@tool
def get_restaurant_recommendation(city: str, cuisine: str) -> dict:
    """Get restaurant recommendations for a specific city and cuisine type."""
    
    city_map = {
        "san francisco": "sf",
        "sf": "sf",
        "new york": "nyc",
        "nyc": "nyc",
        "new york city": "nyc",
        "miami": "miami",
        "ny": "nyc"
    }
    
    city_key = city_map.get(city.lower(), city.lower())
    
    recommendations = {
        "sf:italian": {
            "restaurant_name": "Tony's Pizza Napoletana",
            "price_range": "$$",
            "description": "Best authentic Italian in North Beach",
            "full_text": "Tony's Pizza Napoletana - Best authentic Italian in North Beach"
        },
        "sf:mexican": {
            "restaurant_name": "La Taqueria",
            "price_range": "$",
            "description": "Famous for mission-style burritos",
            "full_text": "La Taqueria - Famous for mission-style burritos"
        },
        "sf:chinese": {
            "restaurant_name": "R&G Lounge",
            "price_range": "$$",
            "description": "Award-winning Cantonese seafood",
            "full_text": "R&G Lounge - Award-winning Cantonese seafood"
        },
        "nyc:italian": {
            "restaurant_name": "Carbone",
            "price_range": "$$$",
            "description": "Classic Italian-American fine dining",
            "full_text": "Carbone - Classic Italian-American fine dining"
        },
        "nyc:mexican": {
            "restaurant_name": "Casa Enrique",
            "price_range": "$$",
            "description": "Michelin-starred Mexican in NYC",
            "full_text": "Casa Enrique - Michelin-starred Mexican in NYC"
        },
        "nyc:chinese": {
            "restaurant_name": "Joe's Shanghai",
            "price_range": "$$",
            "description": "Famous for soup dumplings",
            "full_text": "Joe's Shanghai - Famous for soup dumplings"
        },
        "nyc:japanese": {
            "restaurant_name": "Ippudo NY",
            "price_range": "$$",
            "description": "Legendary ramen shop",
            "full_text": "Ippudo NY - Legendary ramen shop"
        },
        "miami:cuban": {
            "restaurant_name": "Versailles Restaurant",
            "price_range": "$$",
            "description": "The most famous Cuban restaurant in Miami",
            "full_text": "Versailles Restaurant - The most famous Cuban restaurant in Miami"
        },
        "miami:seafood": {
            "restaurant_name": "Joe's Stone Crab",
            "price_range": "$$$",
            "description": "Historic seafood institution since 1913",
            "full_text": "Joe's Stone Crab - Historic seafood institution since 1913"
        },
        "miami:italian": {
            "restaurant_name": "Macchialina",
            "price_range": "$$",
            "description": "Rustic Italian in South Beach",
            "full_text": "Macchialina - Rustic Italian in South Beach"
        }
    }
    
    key = f"{city_key}:{cuisine.lower()}"
    print(f"DEBUG: Buscando clave '{key}'")
    
    if key in recommendations:
        data = recommendations[key]
        return {
            "city": city,
            "cuisine": cuisine,
            "restaurant_name": data["restaurant_name"],
            "price_range": data["price_range"],
            "description": data["description"],
            "full_text": data["full_text"]
        }
    else:
        print(f"DEBUG: Clave '{key}' no encontrada, usando default")
        return {
            "city": city,
            "cuisine": cuisine,
            "restaurant_name": "Local favorite",
            "price_range": "$$",
            "description": f"Great {cuisine} restaurant in {city}",
            "full_text": f"Great {cuisine} restaurants in {city} include: Local favorite, Popular spot, and Hidden gem"
        }


@dataclass
class UserContext:
    """Custom runtime context schema."""
    user_id: str
    user_name: str = "Guest"
    favorite_cuisine: str = "italian"


@tool
def get_user_cuisine_preference(runtime: ToolRuntime[UserContext]) -> str:
    """Retrieve user's preferred cuisine based on user ID."""
    user_id = runtime.context.user_id
    
    preferences = {
        "1": "italian",
        "2": "mexican",
        "3": "chinese",
        "4": "japanese",
        "5": "indian",
        "alice": "italian",
        "bob": "mexican",
        "charlie": "chinese"
    }
    
    cuisine = preferences.get(user_id, preferences.get(user_id.lower(), "italian"))
    runtime.context.favorite_cuisine = cuisine

    print(f"DEBUG: Usuario {user_id} prefiere cocina {cuisine}")
    
    return cuisine


print("Herramientas de restaurantes creadas correctamente")

Herramientas de restaurantes creadas correctamente


### Real world model configuration

In [40]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "google_genai:gemini-2.5-flash-lite",  
    temperature=0.7,
    timeout=15,
    max_tokens=800
)
print("Modelo configurado con temperatura más alta para creatividad")

Modelo configurado con temperatura más alta para creatividad


### Define new answer format

In [41]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class RestaurantResponse:
    """Response schema for restaurant recommendations."""
    punny_recommendation: str
    restaurant_name: Optional[str] = None
    cuisine_type: Optional[str] = None
    city: Optional[str] = None
    price_range: Optional[str] = None

print("Nuevo formato RestaurantResponse creado")

Nuevo formato RestaurantResponse creado


### Memory Configuration

In [42]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()
print("Memoria configurada")

Memoria configurada


### Restaurants creation Agent

In [43]:
from langchain.agents.structured_output import ToolStrategy

agent = create_agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[get_user_cuisine_preference, get_restaurant_recommendation],
    context_schema=UserContext,
    response_format=ToolStrategy(RestaurantResponse),
    checkpointer=checkpointer
)

print("Agente de recomendaciones de restaurantes creado")

Agente de recomendaciones de restaurantes creado


## Tests for the new Agent

### Test with user1  (italian)

In [44]:
config = {"configurable": {"thread_id": "foodie-1"}}

response = agent.invoke(
    {"messages": [{"role": "user", "content": "I'm hungry, where should I eat in San Francisco?"}]},
    config=config,
    context=UserContext(user_id="1", user_name="Alice")
)

print("USUARIO 1 (Alice - italiana):")
print(response['structured_response'])

DEBUG: Usuario 1 prefiere cocina italian
DEBUG: Buscando clave 'sf:italian'
USUARIO 1 (Alice - italiana):
RestaurantResponse(punny_recommendation="I'm sure you'll have a *tony* good time at Tony's Pizza Napoletana!", restaurant_name="Tony's Pizza Napoletana", cuisine_type='italian', city='San Francisco', price_range='$$')


### Same Conversations, to prove memory

In [45]:
import time

response2 = agent.invoke(
    {"messages": [{"role": "user", "content": "what about something cheaper?"}]},
    config=config,
    context=UserContext(user_id="1", user_name="Alice")
)

print( "SEGUNDA CONSULTA (más económico):")
print(response2['structured_response'])
print(f"\nPunny recommendation: {response2['structured_response'].punny_recommendation}")

DEBUG: Buscando clave 'sf:italian'
SEGUNDA CONSULTA (más económico):
RestaurantResponse(punny_recommendation="I can't find anything cheaper that's still *pasta*-bly good. Maybe try a different city or cuisine?", restaurant_name=None, cuisine_type='italian', city='San Francisco', price_range='$')

Punny recommendation: I can't find anything cheaper that's still *pasta*-bly good. Maybe try a different city or cuisine?
